In [2]:
import os

# nvidia-smi 기준 물리 GPU 3번만 노출
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [3]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import torch

model_id = "google/owlv2-base-patch16-ensemble"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(DEVICE)

The image processor of type `Owlv2ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/418 [00:00<?, ?it/s]

이제 이미지를 불러오고 텍스트 프롬프트를 작성해보자! 여기서 바깥쪽 리스트는 객체를 배치 처리하는 데 사용되므로, 퀘스트 쿼리에는 바깥쪽 차원이 생김.  
텍스트 프롬프트와 이미지를 프로세서에 전달하자!

In [4]:
import requests
from PIL import Image

image_url = "https://huggingface.co/datasets/vlmbook/images/resolve/main/bee.jpg"
image = Image.open(requests.get(image_url, stream=True).raw)
queris = [["bee on the flower", "bee in the hive"]]
inputs = processor(images=image, text=queris, return_tensors="pt").to(DEVICE)

In [5]:
sizes = [image.size[::-1]]
outputs = model(**inputs)
results = processor.post_process_grounded_object_detection(outputs, threshold=0.3, target_sizes=sizes, text_labels=queris)

In [6]:
for box, score, label in zip(results[0]["boxes"], results[0]["scores"], results[0]["text_labels"]):
    box = [round(coord, 2) for coord in box.tolist()]
    print(
        f"Detected {label} with confidence "
        f"{round(score.item(), 3)} at location {box}"
    )

Detected bee on the flower with confidence 0.336 at location [1249.42, 591.11, 3768.74, 3193.21]
Detected bee on the flower with confidence 0.336 at location [2346.75, 1623.07, 2845.86, 2115.57]
Detected bee on the flower with confidence 0.311 at location [4517.6, 2404.58, 4974.45, 3015.79]


In [7]:
top_k = 10

res = processor.post_process_grounded_object_detection(
    outputs,
    threshold=0.01,
    target_sizes=[image.size[::-1]],
)

res[0]["scores"] = res[0]["scores"][:top_k]
res[0]["labels"] = res[0]["labels"][:top_k]
res[0]["boxes"] = res[0]["boxes"][:top_k]